# File Breakdown

If you would like to best understand this file especially code block outputs, please navigate it alongside the appropriate documentation via the PC: Data Documentation google docs file. Otherwise you can do without it only reading the comments and ignoring the notes, but if you do not understand an output or are beginning to feel confused please refer to the documentation afterall that's what it's there for.


Navigate to the correct section of the google docs file by:
- Opening the panel on the left
- Clicking on Kayden
- Clicking on WorldBankForecasting (under Forecasting)

There will be notes in the comments for when to refer to the documentation for further clarity. Notes in all caps are changes I may need to make

Hopefully I cooked with this lol

### Executive Summary:
Step 1: Data Loading (Read in dataset, Import packages, etc.)

Step 2: Exploratory Data Analysis (Identify null values, Plot random countries to gauge trends we’re working with)

Step 3: Data Preprocessing (Get rid of null values, Get rid of null countries / years where no data was collected)

Step 4: Model Building (Holt-Winters, ARIMA)

Step 5: Model Evaluation (Use MAE, RMSE, MAPE (for interpretability?, Create confidence intervals?)

Step 6: Conclusion

# Step 1: Data Loading

In [1]:
# Import packages and warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", message="Non-stationary starting autoregressive parameters")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.simplefilter('ignore', ConvergenceWarning)

In [ ]:
import sys
import os
from pathlib import Path

# Move up from 'src/processing' to the main 'PC-Data-Dash' folder
# Change the number of '.parent' calls depending on where your notebook is
root_path = str(Path.cwd().parent.parent) 

if root_path not in sys.path:
    sys.path.append(root_path)

from src.pipeline.utils import project_root
project_root = project_root()
df = pd.read_csv(project_root / 'data/interim/world_bank_interim.csv')
df.head()

: 

: 

# Step 2: Exploratory Data Analysis

In [ ]:
# Identifies how many rows and columns in dataset, if there are any null entries in each column and dataset memory size 
df.info()
# Prints number of null entries per column
# See Note 2.1 in documentation
df.isnull().sum()

: 

: 

In [ ]:
# Statistical breakdown of numerical data
df.describe()

: 

: 

In [ ]:
# Print the rows with no score values provided
# See Note 2.2 in documentation
null_rows = df[df['value'].isnull()]
print(null_rows)

: 

: 

In [ ]:
# See Note 2.3 in documentation
for col in null_rows:
    null_breakdown = df[df['value'].isnull()][col].value_counts()
    print(null_breakdown)

: 

: 

In [ ]:
# Prints the number of years with null scores excluding 2024 for each country
select_null_rows = (df[(df['year'] != 2024) & (df['value'].isnull())])
print(select_null_rows['country'].value_counts())
print((select_null_rows['country'].value_counts()).count())

: 

: 

In [ ]:
# Create a list of countries that have null scores that aren't in 2024 (used because the rest of these are not countries which will be purged later)
exceptioncountries = ['Curacao', 'St. Martin (French part)', 'Sint Maarten (Dutch part)', 'South Sudan', 'Channel Islands', 'Sudan', 'World']

: 

: 

Note: Drop rows with 2024 data, drop Kosovo (iso XKS) because all 15 years are dataless.

In [ ]:
# THIS NEEDS TO BE TWEAKED BECAUSE OUTLIER CALCULATIONS ARE FOR 2010 INSTEAD OF THE MOST RECENT YEAR
# Outlier Identification (So we know who's being affected the most by the given variable)
# Print the lower bound, median and upper bound of the scores and the country outliers
outliers = []
visited_countries = []
Q1 = df['value'].quantile(0.25)
Q2 = df['value'].quantile(0.50)
Q3 = df['value'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
for index, row in df.iterrows():
    if row['country'] in visited_countries:
        continue 
    if (row['value'] < lower_bound) | (row['value'] > upper_bound):
        outliers.append({
            'country': row.country,
            'value': row.value
        })
        visited_countries.append(row['country'])

df_outliers = pd.DataFrame(outliers)
print(lower_bound, Q2, upper_bound)
print(df_outliers)

: 

: 

In [ ]:
# THIS MAY BE COMPLETELY IRRELEVANT LETS SEE
# Prints how many years outlier countries have been outliers for ()
outliers = []
visited_countries = []
Q1 = df['value'].quantile(0.25)
Q2 = df['value'].quantile(0.50)
Q3 = df['value'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
for index, row in df.iterrows():
    if (row['value'] < lower_bound) | (row['value'] > upper_bound):
        outliers.append({
            'country' : row.country,
            'year' : row.year,
            'value': row.value
        })
        visited_countries.append(row['country'])

df_outliers = pd.DataFrame(outliers)
print(df_outliers['country'].value_counts())

: 

: 

In [ ]:
# THIS MAY ALSO BE IRRELEVANT
# Visualize population density graphs for countries that weren't an outlier for all years in case they dropped off
country = ['Israel', 'Comoros', 'Burundi', 'India', 'Rwanda']
for i in country:
    sns.lineplot(data=df[df['country'] == i], x='year', y='value')
    plt.ylabel('Population Density')
    plt.title(f"{i} Population Density Over Time")
    plt.show()

: 

: 

# Step 3: Data Preprocessing

In [ ]:
# Drop null rows
# Drop non country rows
# See Note 3.1 in documentation
df = df.drop(columns=["indicator"])
for index, row in df.iterrows():
    if (row['year'] == 2024) or (row['country'] in select_null_rows['country'].values and row['country'] not in exceptioncountries):
        df = df.drop(index)


: 

: 

In [ ]:
# Look at our new cleaned dataframe
# See Note 3.2 in documentation
df.describe()

: 

: 

In [ ]:
# See what the new dataframe is looking like
df.head()

: 

: 

# Step 4: Model Building

1. Fit versions of Holt-Winters model
2. Fit versions of ARIMA model (probably just to ensure Holt-Winters is on the right track)

## 4.1: Naive model

In [ ]:
results_list_naive = []
for country in df["country"].unique():
    df_subset = df[df["country"] == country].sort_values("year").dropna(subset=["value"])
    if len(df_subset) < 10: continue # Need enough history

    # We want to check the last 3 years: e.g., 2022, 2023, 2024
    for i in range(1, 4): # -1, -2, -3
        actual = df_subset["value"].iloc[-i]
        actual_year = df_subset["year"].iloc[-i]
        
        # The prediction comes from 6 years PRIOR to the actual year
        predicted = df_subset["value"].iloc[-i - 6]
        
        percent_error = (abs(actual - predicted) / actual) * 100
        results_list_naive.append({
            "country": country, "year": actual_year, "forecast": predicted,
            "actual": actual, "percent_error": percent_error
        })

final_df_naive = pd.DataFrame(results_list_naive)
print(f"Naive Median Error: {final_df_naive['percent_error'].median():.2f}%")

: 

: 

## 4.2: Holt-Winters Model

In [ ]:
results_list_hw = []
for country in df["country"].unique():
    df_subset = df[df["country"] == country].sort_values("year").dropna(subset=["value"])
    if len(df_subset) < 12: continue

    y = df_subset["value"]
    y.index = pd.PeriodIndex(df_subset["year"], freq="Y")

    for i in range(1, 4): # Checking last 3 years
        # Train on data ending 6 years before the target year
        y_train = y.iloc[: -i - 5] # -6, -7, -8 relative to end
        actual = y.iloc[-i]
        actual_year = y.index[-i]

        fit = ExponentialSmoothing(y_train, trend="add", seasonal=None).fit()
        # Forecast 6 steps ahead
        forecast = fit.forecast(6).iloc[-1] 

        percent_error = (abs(actual - forecast) / actual) * 100
        results_list_hw.append({
            "country": country, "year": actual_year, "forecast": forecast,
            "actual": actual, "percent_error": percent_error
        })

final_df_hw = pd.DataFrame(results_list_hw)
print(f"HW Median Error: {final_df_hw['percent_error'].median():.2f}%")

: 

: 

In [ ]:
results_list_hw_recursive = []

for country in df["country"].unique():
    df_subset = df[df["country"] == country].sort_values("year").dropna(subset=["value"])
    if len(df_subset) < 12: 
        continue

    y = df_subset["value"]
    y.index = pd.PeriodIndex(df_subset["year"], freq="Y")

    # Testing the last 3 years (2022, 2023, 2024)
    for i in range(1, 4):
        actual = y.iloc[-i]
        actual_year = y.index[-i]
        
        # 1. INITIAL TRAINING: Train on data ending 6 years before the target
        y_train_initial = y.iloc[: -i - 5] 
        
        # 2. STEP ONE: Predict 3 years into the "middle"
        fit_1 = ExponentialSmoothing(y_train_initial, trend="add", seasonal=None).fit()
        mid_forecasts = fit_1.forecast(3) 
        
        # 3. COMBINE: Add the 3 predicted years to your training data
        # We create a fake series for the predicted years to append
        y_augmented = pd.concat([y_train_initial, mid_forecasts])
        
        # 4. STEP TWO: Re-train on the augmented data and predict 3 more years
        # (Total of 6 years from the original starting point)
        fit_2 = ExponentialSmoothing(y_augmented, trend="add", seasonal=None).fit()
        final_forecast = fit_2.forecast(3).iloc[-1] 

        percent_error = (abs(actual - final_forecast) / actual) * 100
        
        results_list_hw_recursive.append({
            "country": country, 
            "year": actual_year.year, 
            "forecast": final_forecast,
            "actual": actual, 
            "percent_error": percent_error
        })

final_df_hw_rec = pd.DataFrame(results_list_hw_recursive)
print(f"Recursive HW Median Error: {final_df_hw_rec['percent_error'].median():.2f}%")

: 

: 

In [ ]:
country = ['Seychelles', 'Singapore']
for i in country:
    sns.lineplot(data=df[df['country'] == i], x='year', y='value')
    plt.ylabel('Population Density')
    plt.title(f"{i} Population Density Over Time")
    plt.show()

: 

: 

Bruh.

# 4.3: ARIMA model

In [ ]:
results_list_arima = []
for country in df["country"].unique():
    df_subset = df[df["country"] == country].sort_values("year").dropna(subset=["value"])
    if len(df_subset) < 12: continue

    y = df_subset["value"]
    y.index = pd.PeriodIndex(df_subset["year"], freq="Y")

    for i in range(1, 4):
        y_train = y.iloc[: -i - 5] 
        actual = y.iloc[-i]
        actual_year = y.index[-i]

        try:
            model = ARIMA(y_train, order=(1, 1, 0)).fit()
            forecast = model.forecast(steps=6).iloc[-1]
            
            percent_error = (abs(actual - forecast) / actual) * 100
            results_list_arima.append({
                "country": country, "year": actual_year, "forecast": forecast,
                "actual": actual, "percent_error": percent_error
            })
        except:
            continue

final_df_arima = pd.DataFrame(results_list_arima)
print(f"ARIMA Median Error: {final_df_arima['percent_error'].median():.2f}%")

: 

: 

In [ ]:
master_results = []

for country in df["country"].unique():
    df_subset = df[df["country"] == country].sort_values("year").dropna(subset=["value"])
    
    # We need at least 12 years to look back 6 years from a 3-year test window
    if len(df_subset) < 12: 
        continue

    y = df_subset["value"]
    y.index = pd.PeriodIndex(df_subset["year"], freq="Y")

    # Testing 2022, 2023, and 2024
    for i in range(1, 4):
        target_idx = -i
        train_end_idx = -i - 6 
        
        y_train = y.iloc[:train_end_idx + 1]
        actual = y.iloc[target_idx]
        actual_year = y.index[target_idx].year # Convert to integer for easier reading

        # --- Model 1: Naive ---
        naive_pred = y_train.iloc[-1]
        naive_err = (abs(actual - naive_pred) / actual) * 100

        # --- Model 2: Holt-Winters ---
        hw_fit = ExponentialSmoothing(y_train, trend="add", seasonal=None).fit()
        hw_pred = hw_fit.forecast(6).iloc[-1]
        hw_err = (abs(actual - hw_pred) / actual) * 100

        # --- Model 3: ARIMA ---
        try:
            arima_model = ARIMA(y_train, order=(1, 1, 0)).fit()
            arima_pred = arima_model.forecast(steps=6).iloc[-1]
            arima_err = (abs(actual - arima_pred) / actual) * 100
        except:
            arima_err = np.nan # Use NaN if ARIMA fails to converge

        # Store everything in one row
        master_results.append({
            "country": country,
            "year": actual_year,
            "actual": actual,
            "Naive_error": naive_err,
            "HW_error": hw_err,
            "ARIMA_error": arima_err
        })

# THIS is your final_df
final_df = pd.DataFrame(master_results)
display(final_df.head())

: 

: 

In [ ]:
# THIS PART IS NO LONGER RELEVANT 
# Compiled combined version, works best for the python file
# test_list = []

# for country in df["country"].unique():    
#     # 1. Filter and Clean
#     df_subset = df[df["country"] == country].sort_values("year").dropna(subset=["value"])
    
#     # Safety Check
#     if len(df_subset) < 5: # ARIMA and HW like a bit more data
#         continue

#     # 2. IMPORTANT: Re-define your target variables for THIS country
#     y = df_subset["value"]
#     y.index = pd.PeriodIndex(df_subset["year"], freq="Y")
#     y_train = y.iloc[:-1]
#     y_test = y.iloc[-1]
#     actual = y_test

#     # --- MODEL 1: Naive ---
#     naive_forecast = y_train.iloc[-1]
#     naive_per_error = (abs(actual - naive_forecast) / actual) * 100

#     # --- MODEL 2: Holt-Winters (HW) ---
#     hw_fit = ExponentialSmoothing(
#         y_train, trend="add", seasonal=None, initialization_method="estimated"
#     ).fit()
#     hw_forecast = hw_fit.forecast(1).iloc[0]
#     hw_per_error = (abs(actual - hw_forecast) / actual) * 100

#     # --- MODEL 3: ARIMA ---
#     arima_model = ARIMA(y_train, order=(1, 1, 0)).fit()
#     arima_forecast = arima_model.forecast(steps=1).iloc[0]
#     arima_per_error = (abs(actual - arima_forecast) / actual) * 100

#     # 3. Store results
#     output_row = {
#         "country": country,
#         "actual": actual,
#         "Naive_error": naive_per_error,
#         "HW_error": hw_per_error,
#         "ARIMA_error": arima_per_error
#     }
    
#     # MUST BE INDENTED to be inside the loop
#     test_list.append(output_row)

# # Combine and Display
# final_df = pd.DataFrame(test_list)
# display(final_df)

: 

: 

In [ ]:
print(f"Naive Median Error: {final_df_naive['percent_error'].median():.2f}%")
print(f"HW Median Error: {final_df_hw['percent_error'].median():.2f}%")
print(f"ARIMA Median Error: {final_df_arima['percent_error'].median():.2f}%")

: 

: 

In [ ]:
# # SET 2% THRESHOLD?
# # Mapping Outliers for both Models
# Q1_hw = final_df['HW_error'].quantile(0.25)
# Q3_hw = final_df['HW_error'].quantile(0.75)
# IQR_hw = Q3_hw - Q1_hw
# upper_hw = Q3_hw + 1.5 * IQR_hw

# # 2. Define outliers for ARIMA
# Q1_ar = final_df['ARIMA_error'].quantile(0.25)
# Q3_ar = final_df['ARIMA_error'].quantile(0.75)
# IQR_ar = Q3_ar - Q1_ar
# upper_ar = Q3_ar + 1.5 * IQR_ar

# # 3. Create a Filter: Is it an outlier in HW OR ARIMA?
# is_outlier = (final_df['HW_error'] > upper_hw) | (final_df['ARIMA_error'] > upper_ar)

# # 4. Create the final clean report
# outlier_report = final_df[is_outlier][['country', 'HW_error', 'ARIMA_error']].copy()

# # Sort by the highest error to show the "biggest failures" first
# outlier_report = outlier_report.sort_values(by=['ARIMA_error', 'HW_error'], ascending=False)
# print(f"HW threshold {upper_hw}")
# print(f"ARIMA threshold {upper_ar}")
# display(outlier_report)

: 

: 

# Step 5: Model Implementation

In [ ]:
# Compiled combined version, works best for the python file
test_list = []

for country in df["country"].unique():    
    # 1. Filter and Clean
    df_subset = df[df["country"] == country].sort_values("year").dropna(subset=["value"])
    
    # Safety Check
    if len(df_subset) < 5: # ARIMA and HW like a bit more data
        continue

    # 2. IMPORTANT: Re-define your target variables for THIS country
    y = df_subset["value"]
    y.index = pd.PeriodIndex(df_subset["year"], freq="Y")
    y_train = y.iloc[:]

    # --- MODEL 2: Holt-Winters (HW) ---
    hw_fit = ExponentialSmoothing(
        y_train, trend="add", seasonal=None, initialization_method="estimated"
    ).fit()
    hw_forecast = hw_fit.forecast(1).iloc[0]

    # --- MODEL 3: ARIMA ---
    arima_model = ARIMA(y_train, order=(1, 1, 0)).fit()
    arima_forecast = arima_model.forecast(steps=1).iloc[0]

    # 3. Store results
    output_row = {
        "country": country,
        "HW_Prediction": hw_forecast,
        "ARIMA_Prediction": arima_forecast,
        "Weighted_Prediction": (hw_forecast + arima_forecast) / 2
    }
    
    # MUST BE INDENTED to be inside the loop
    test_list.append(output_row)

# Combine and Display
final_df = pd.DataFrame(test_list)
display(final_df)

: 

: 

In [ ]:
future_predictions = []

for country in df["country"].unique():
    # 1. Clean and Prepare
    df_subset = df[df["country"] == country].sort_values("year").dropna(subset=["value"])
    
    # Need enough history to train ARIMA(1,1,0)
    if len(df_subset) < 5: 
        continue

    y = df_subset["value"]
    y.index = pd.PeriodIndex(df_subset["year"], freq="Y")

    try:
        # 2. Train on ALL available data
        model = ARIMA(y, order=(1, 1, 0)).fit()
        
        # 3. Forecast 6 years ahead (2024, 2025, 2026, 2027, 2028, 2029)
        # Assuming your data ends in 2023
        forecast_series = model.forecast(steps=6)
        
        # 4. Map the forecasts to the specific years
        # forecast_series[3] = 2027, [4] = 2028, [5] = 2029
        years = [2027, 2028, 2029]
        forecast_values = forecast_series.iloc[3:].values 

        for year, val in zip(years, forecast_values):
            future_predictions.append({
                "country": country,
                "year": year,
                "predicted_value": round(val, 4)
            })
            
    except Exception as e:
        continue

# 5. Output the results
final_predictions_df = pd.DataFrame(future_predictions)
print(final_predictions_df)

: 

: 

In [ ]:
final_df_scaled.describe()

: 

: 

: 

: 

Convert finalized into .csv